## 랭체인 Multi-Chain 구현
- LCEL(LangChain Expression Language)로 체인 만들기

In [1]:
import os
from dotenv import load_dotenv

# .env 파일의 내용 불러오기
load_dotenv("C:/env/.env")

True

#### 순차 연결 (Pipeline with | Operator)

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# LLM 모델 선택
llm = ChatOpenAI(model="gpt-4o-mini")

# 1단계: 텍스트 요약 프롬프트
summary_prompt = PromptTemplate.from_template(
    "다음 텍스트를 한 문장으로 요약해줘:\n{text}"
)

# 2단계: 영어 번역 프롬프트
translate_prompt = PromptTemplate.from_template(
    "다음 문장을 영어로 번역:\n{summary}"
)

# 체인 구성: 요약 → 번역
multi_chain = summary_prompt | llm | translate_prompt | llm

# 실행
result = multi_chain.invoke({"text": "랭체인은 LLM 애플리케이션을 쉽게 만들 수 있는 프레임워크이다."})
print(result.content)

The sentence translates to: "LangChain is a framework that makes it easy to create LLM applications."


#### 조건 분기 (Router / Multi-Prompt Chain)

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableBranch
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

# 체인 정의
math_chain = PromptTemplate.from_template("계산 문제 풀기: {input}") | llm | parser
translate_chain = PromptTemplate.from_template("영어로 번역: {input}") | llm | parser
default_chain = PromptTemplate.from_template("대화 응답: {input}") | llm | parser

# 간단한 분기 조건
router = RunnableBranch(
    (lambda x: "계산" in x["input"], math_chain),
    (lambda x: "번역" in x["input"], translate_chain),
    default_chain
)

# 실행 예시
print(router.invoke({"input": "2+2 계산해줘"}))      # → math_chain 실행
print(router.invoke({"input": "이 문장 번역해줘"}))  # → translate_chain 실행
print(router.invoke({"input": "안녕, 오늘 어때?"}))   # → default_chain 실행


2 + 2 = 4입니다.
Sure! The translation of "이 문장 번역해줘" in English is "Please translate this sentence."
안녕! 나는 항상 좋지. 너는 오늘 어때?


#### ParallelChain (병렬 실행)

In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

llm = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

# 각 작업 체인 정의
summary_chain = PromptTemplate.from_template("다음 텍스트를 한 문장으로 요약해줘:\n{text}") | llm | parser
sentiment_chain = PromptTemplate.from_template("다음 텍스트의 감정을 분석해줘:\n{text}") | llm | parser
keyword_chain = PromptTemplate.from_template("다음 텍스트에서 키워드 3개만 뽑아줘:\n{text}") | llm | parser

# 병렬 실행 체인
parallel_chain = RunnableParallel(
    summary=summary_chain,
    sentiment=sentiment_chain,
    keywords=keyword_chain
)

# 실행
result = parallel_chain.invoke({"text": "LangChain은 LLM 애플리케이션 개발을 쉽게 해주는 프레임워크이다."})
                                 # 이 입력이 병렬 체인의 모든 서브 체인(summary, sentiment, keywords)에 동시에 전달.
                                 # 각 체인이 끝날 때까지 기다리지 않고 한꺼번에 결과를 요청한다.

print(result)


{'summary': 'LangChain은 대규모 언어 모델(LLM) 애플리케이션 개발을 간편하게 지원하는 프레임워크이다.', 'sentiment': '해당 텍스트는 중립적인 감정을 지니고 있습니다. "LangChain은 LLM 애플리케이션 개발을 쉽게 해주는 프레임워크이다."라는 문장은 LangChain의 기능과 장점을 설명하고 있으며, 긍정적인 평가나 부정적인 감정이 드러나지 않습니다. 단순히 정보를 전달하는 내용으로 구성되어 있습니다.', 'keywords': '1. LangChain\n2. LLM\n3. 애플리케이션 개발'}


In [5]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o-mini")

from langchain_core.messages import HumanMessage, SystemMessage


messages = [
    SystemMessage(content="너는 춘향전의 주인공 성춘향이야. 그 캐릭터에 맞게 답변해."),
    HumanMessage(content="나는 남원 고을 변사또야. 오늘 저녁 나와 함께 잔치를 즐기지 않겠는가?"),
]

model.invoke(messages)

AIMessage(content='변사또님, 이렇게 초대해 주셔서 감사합니다. 하지만 저는 사랑하는 이몽룡과의 약속이 있기에 그 자리에는 참석할 수 없답니다. 제 마음은 오직 그를 향하고 있으니까요. 잔치가 즐거웠으면 좋겠습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 65, 'total_tokens': 129, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ced0f95ffb', 'id': 'chatcmpl-E5o87ymQnXTBn45NFE9kgcVY0DaEE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f9d73-e9ff-77f0-8bae-a8857b3a3f41-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 65, 'output_tokens': 64, 'total_tokens': 129, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### 출력 파서(output parser) 사용하기  

In [6]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()   # 텍스트만 추출하여 반환하는 파서 객체 생성

result = model.invoke(messages)
parser.invoke(result)  # 문자열(content)만 출력 

'변사또님, 이렇게 초대해 주시니 감사합니다. 하지만 저 성춘향은 사랑하는 이몽룡을 기다리고 있습니다. 잔치는 즐겁고 기쁜 일이지만, 제 마음은 변하지 않을 것 같습니다. 어떠신가요? 그럼에도 불구하고 함께 즐겨주신다면 다소 기쁘겠지만, 제 마음은 그리 옮기기 어려울 것 같아요.'

### 파이프 연산자("|") 사용하기

In [7]:
chain = model | parser
chain.invoke(messages)

'변사또님, 제가 당신과 함께 잔치를 즐기는 것은 제 마음의 바람이 아니옵니다. 저 성춘향은 진실한 사랑을 믿고, 그 마음을 저버릴 수 없으니 양반의 잔치에 참석할 수는 없사옵니다. 제 마음은 오로지 이도령에게 있사옵니다. 이해해 주시기를 부탁드리옵니다.'

### 프롬프트 템플릿 사용하기

In [8]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {story}에 나오는 {character_a} 역할이다. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕? 나는 {character_b}입니다. 오늘 시간 괜찮으시면 {activity}에 같이 갈까요?"

prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

result = prompt_template.invoke({
    "story": "춘향전",
    "character_a": "성춘향",
    "character_b": "변사또",
    "activity": "잔치"
})

print(result)

messages=[SystemMessage(content='너는 춘향전에 나오는 성춘향 역할이다. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕? 나는 변사또입니다. 오늘 시간 괜찮으시면 잔치에 같이 갈까요?', additional_kwargs={}, response_metadata={})]


In [9]:
chain = prompt_template | model | parser

chain.invoke({
    "story": "춘향전",
    "character_a": "성춘향",
    "character_b": "변사또",
    "activity": "잔치"
})

'안녕하세요, 변사또님. 잔치에 초대해 주셔서 고맙습니다. 하지만 전 물러날 수 없어요. 사랑하는 이도 있고, 제 신분에 맞지 않는 일이라서요. 변사또님의 마음은 고맙지만, 다른 길을 걷는 것이 좋을 것 같아요.'

In [10]:
chain = prompt_template | model | parser

chain.invoke({
    "story": "춘향전",
    "character_a": "성춘향",
    "character_b": "이몽룡",
    "activity": "잔치"
})

'안녕하세요, 몽룡님! 이렇게 만나니 반갑습니다. 잔치라니, 정말 즐거운 시간이 될 것 같아요. 함께 가면 좋겠어요. 그런데 혹시 어떤 잔치인지 궁금하네요. 설명해 주실 수 있나요?'

In [11]:
chain = prompt_template | model | parser

chain.invoke({
    "story": "춘향전",
    "character_a": "향단이",
    "character_b": "방자",
    "activity": "잔치"
})

'안녕하세요, 방자님! 잔치라니 참 재미있겠어요. 같이 가지요, 방자님과 함께라면 더욱 즐거운 시간이 될 것 같아요! 어느 잔치인지 궁금한데, 어디서 열리는 건가요?'

In [12]:
#  프롬프트 템플릿/파이프 연산자/출력 파서 함께 연동
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 1. 프롬프트 템플릿 정의
prompt_template = ChatPromptTemplate.from_template(
    "당신은 친절한 한국어 선생님입니다. "
    "사용자의 질문에 간단히 답하세요.\n\n질문: {question}"
)

# 2. 모델 정의
model = ChatOpenAI(model="gpt-4o-mini")

# 3. 출력 파서 정의 (문자열만 추출)
parser = StrOutputParser()

# 4. 파이프라인 구성
chain = prompt_template | model | parser

# 5. 실행
result = chain.invoke({"question": "한글은 몇 년도에 창제되었나요?"})
print(result)

한글은 1443년에 창제되었고, 1446년에 반포되었습니다.
